In [4]:
# notebooks/05_feature_engineering.ipynb

import sys
for mod in list(sys.modules.keys()):
    if mod in ('config','data_loader','preprocessor','trainer','evaluator'):
        del sys.modules[mod]
sys.path.insert(0, '../src')

import warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from data_loader import (load_raw_data, clean_dependents, handle_credit_history,
                          encode_target, engineer_features, get_X_y,
                          get_train_val_split)
from preprocessor import build_preprocessor
from trainer      import build_full_pipeline, get_models
from evaluator    import evaluate_all_models
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# ── Load and engineer ────────────────────────────────────────────────────────
df, _ = load_raw_data()
df    = clean_dependents(df)
df    = handle_credit_history(df)
df    = encode_target(df)
df    = engineer_features(df)        # ← new step

X, y  = get_X_y(df)
X_train, X_val, y_train, y_val = get_train_val_split(X, y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Compare: without vs with engineered features ─────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.impute   import SimpleImputer

results = []
for use_eng in [False, True]:
    label = "With engineering" if use_eng else "Without engineering"
    print(f"\nTraining: {label}")

    pre      = build_preprocessor(use_engineered=use_eng)
    rf       = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    pipeline = build_full_pipeline(rf)

    # Rebuild pipeline with correct preprocessor
    from sklearn.pipeline import Pipeline as SKPipeline
    pipeline = SKPipeline([("preprocessor", pre), ("classifier", rf)])

    cv_auc = cross_val_score(pipeline, X_train, y_train,
                              cv=skf, scoring='roc_auc', n_jobs=-1)
    pipeline.fit(X_train, y_train)

    from sklearn.metrics import roc_auc_score
    val_auc = roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1])

    results.append({
        "config":   label,
        "cv_auc":   round(cv_auc.mean(), 4),
        "cv_std":   round(cv_auc.std(),  4),
        "val_auc":  round(val_auc,       4),
    })
    print(f"  CV AUC:  {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
    print(f"  Val AUC: {val_auc:.4f}")

print("\n── Feature Engineering Impact ──")
print(pd.DataFrame(results).set_index("config").to_string())


Training: Without engineering
  CV AUC:  0.7607 ± 0.0693
  Val AUC: 0.7977

Training: With engineering
  CV AUC:  0.7528 ± 0.0689
  Val AUC: 0.8563

── Feature Engineering Impact ──
                     cv_auc  cv_std  val_auc
config                                      
Without engineering  0.7607  0.0693   0.7977
With engineering     0.7528  0.0689   0.8563


In [5]:
# Which engineered features correlate with target?
eng_cols = ['TotalIncome', 'LoanAmountPerIncome', 'IncomePerDependent',
            'EMI_to_Income_ratio', 'IsHighIncome',
            'IsSelfEmployedHighLoan', 'HasCoapplicant']

corr = df[eng_cols + ['Loan_Status']].corr()['Loan_Status'].drop('Loan_Status')
print(corr.sort_values(key=abs, ascending=False).round(4))

LoanAmountPerIncome      -0.0766
HasCoapplicant            0.0752
TotalIncome              -0.0313
EMI_to_Income_ratio      -0.0310
IncomePerDependent       -0.0219
IsHighIncome             -0.0149
IsSelfEmployedHighLoan    0.0121
Name: Loan_Status, dtype: float64


In [3]:
import sys
for mod in list(sys.modules.keys()):
    if mod in ('config', 'data_loader', 'preprocessor'):
        del sys.modules[mod]
sys.path.insert(0, '../src')

import numpy as np
from data_loader  import (load_raw_data, clean_dependents,
                           handle_credit_history, encode_target,
                           engineer_features, get_X_y, get_train_val_split)
from preprocessor import build_preprocessor

df, _ = load_raw_data()
df    = clean_dependents(df)
df    = handle_credit_history(df)
df    = encode_target(df)
df    = engineer_features(df)

X, y  = get_X_y(df)
X_train, X_val, y_train, y_val = get_train_val_split(X, y)

# Test both modes
for use_eng in [False, True]:
    pre   = build_preprocessor(use_engineered=use_eng)
    X_out = pre.fit_transform(X_train)
    names = pre.get_feature_names_out()
    label = "WITH engineering" if use_eng else "WITHOUT engineering"
    print(f"\n{label}")
    print(f"  Output shape : {X_out.shape}")
    print(f"  Features     : {len(names)}")
    print(f"  Any NaN      : {np.isnan(X_out).any()}")
    print(f"  Any Inf      : {np.isinf(X_out).any()}")
    ch = any('Credit_History' in n for n in names)
    print(f"  Credit_History present: {ch}")

print("\n✓ preprocessor.py verified")


WITHOUT engineering
  Output shape : (491, 12)
  Features     : 12
  Any NaN      : False
  Any Inf      : False
  Credit_History present: True

WITH engineering
  Output shape : (491, 21)
  Features     : 21
  Any NaN      : False
  Any Inf      : False
  Credit_History present: True

✓ preprocessor.py verified
